# Build Step 1 — Registry Schema & Seeds (mongosh)

**Framework:** File Validation & Routing (`fvr` database) • **Deliverable:** the 7 Phase-1 registry collections with `$jsonSchema` validators + indexes + seed data • **Checkpoint:** R1 — you sign off on contracts & seeds before any application code (Step 3+)

## How to run

1. Requires a **replica set** (Atlas or self-managed — change streams + transactions need it; single-node dev replica set is fine).
2. Run each fenced block in `mongosh` against `use fvr`, in cell order. All blocks are **re-runnable**: `createCollection` is guarded with `if (!db.getCollectionInfos(...))`, indexes are `createIndex` (idempotent), and seeds use `insertMany({ordered:false})` — on re-run, duplicate-key errors on the unique keys are **expected and harmless**.
3. Placeholders to edit before running (flagged `// EDIT`):
     * `volume_root` paths (Phase-1 open question #5 — volume roots not yet confirmed)
     * `bootstrap admin email` (one-time first-admin insert — comment it out afterwards)
     * `schema_versions.fields` column lists — **RESOLVED at R1:** sourced from the real `mdd_platform` silver schemas. Note `ft_billing` was renamed **`ft_payments`** to match the actual domain (src_003/src_008 payments).

## Scope

| # | Collection | Validator | Indexes | Seeds |
| --- | --- | --- | --- | --- |
| 1 | `users` | ✅ | `email` unique | optional bootstrap admin |
| 2 | `source_systems` | ✅ | `source_id` unique | `src_portal` |
| 3 | `file_types` | ✅ | `code` unique | 5 types |
| 4 | `schema_versions` | ✅ | `(file_type_id, version)` unique + **partial unique: one `active` per file type** | 5 × v1 (**real fields from mdd_platform**) |
| 5 | `validation_rules` | ✅ | `(rule_code, version)` unique | 8 rules |
| 6 | `routing_policies` | ✅ | `priority` | 3 policies |
| 7 | `volume_map` | ✅ | `(file_type_id, destination_type)` unique | 7 rows (5 intake + `*` quarantine + `*` staging) |

**Not in Step 1:** operational collections (`event_log`, `upload_sessions`, `file_registry`, `validation_results`, `file_transfers`, `file_lifecycle`, `audit_log`, `notifications`) — that is build **Step 2**, which carries the LLD v1.1–v1.3 field additions into their validators. `retention_policies`/`archive_jobs` are Phase 3.

**Field reference:** `Database Design LLD` v1.3 is the authoritative source for every field below.

## 1. DDL — `users`, `source_systems`, `file_types`

```javascript
use fvr

// ---------- users ----------
if (!db.getCollectionInfos({name: "users"}).length) {
  db.createCollection("users", {
    validator: { $jsonSchema: {
      bsonType: "object",
      required: ["user_id", "email", "display_name", "roles", "is_active", "created_at"],
      properties: {
        user_id:       { bsonType: "string", pattern: "^usr_" },
        email:         { bsonType: "string", minLength: 5 },
        display_name:  { bsonType: "string", minLength: 1 },
        roles:         { bsonType: "array", minItems: 1,
                         items: { enum: ["UPLOADER", "OPS", "ADMIN"] } },
        department:    { bsonType: ["string", "null"] },
        is_active:     { bsonType: "bool" },
        created_at:    { bsonType: "date" },
        last_login_at: { bsonType: ["date", "null"] }
      }
    }},
    validationLevel: "strict", validationAction: "error"
  });
}
db.users.createIndex({ user_id: 1 }, { unique: true });
db.users.createIndex({ email: 1 },   { unique: true });

// ---------- source_systems ----------
if (!db.getCollectionInfos({name: "source_systems"}).length) {
  db.createCollection("source_systems", {
    validator: { $jsonSchema: {
      bsonType: "object",
      required: ["source_id", "name", "source_type", "connection_ref", "poll_or_push", "is_active"],
      properties: {
        source_id:        { bsonType: "string", pattern: "^src_" },
        name:             { bsonType: "string", minLength: 1 },
        source_type:      { enum: ["PORTAL", "SHAREPOINT", "SFTP", "DB_VOLUME"] },
        connection_ref:   { bsonType: "string", minLength: 1 },   // vault reference only — never credentials
        poll_or_push:     { enum: ["PUSH", "POLL"] },
        detection_config:{ bsonType: ["object", "null"] },
        is_active:        { bsonType: "bool" }
      }
    }},
    validationLevel: "strict", validationAction: "error"
  });
}
db.source_systems.createIndex({ source_id: 1 }, { unique: true });

// ---------- file_types ----------
if (!db.getCollectionInfos({name: "file_types"}).length) {
  db.createCollection("file_types", {
    validator: { $jsonSchema: {
      bsonType: "object",
      required: ["file_type_id", "code", "allowed_extensions", "header_row", "is_active", "version"],
      properties: {
        file_type_id:       { bsonType: "string", pattern: "^ft_" },
        code:               { bsonType: "string", minLength: 2 },
        description:        { bsonType: ["string", "null"] },
        allowed_extensions: { bsonType: "array", minItems: 1,
                              items: { bsonType: "string", pattern: "^\\." } },
        max_file_size_mb:   { bsonType: ["int", "null"], minimum: 1 },
        min_rows:           { bsonType: ["int", "null"], minimum: 0 },
        encoding:           { bsonType: ["string", "null"] },
        delimiter:          { bsonType: ["string", "null"] },
        header_row:         { bsonType: "bool" },
        identification:     { bsonType: ["object", "null"] },
        is_active:          { bsonType: "bool" },
        version:            { bsonType: "int", minimum: 1 }
      }
    }},
    validationLevel: "strict", validationAction: "error"
  });
}
db.file_types.createIndex({ file_type_id: 1 }, { unique: true });
db.file_types.createIndex({ code: 1 },           { unique: true });
```

## 2. DDL — `schema_versions`, `validation_rules`

```javascript
// ---------- schema_versions ----------
if (!db.getCollectionInfos({name: "schema_versions"}).length) {
  db.createCollection("schema_versions", {
    validator: { $jsonSchema: {
      bsonType: "object",
      required: ["schema_id", "file_type_id", "version", "status", "effective_from", "fields", "created_by", "created_at"],
      properties: {
        schema_id:      { bsonType: "string", pattern: "^sch_" },
        file_type_id:   { bsonType: "string", pattern: "^ft_" },
        version:        { bsonType: "int", minimum: 1 },
        status:         { enum: ["draft", "active", "retired"] },
        effective_from: { bsonType: "date" },
        fields:         { bsonType: "array", minItems: 1, items: { bsonType: "object",
                          required: ["order", "name", "data_type", "nullable"],
                          properties: {
                            order:  { bsonType: "int", minimum: 1 },
                            name:   { bsonType: "string", minLength: 1 },
                            data_type: { enum: ["string", "integer", "decimal", "date", "boolean", "timestamp"] },
                            nullable:  { bsonType: "bool" },
                            max_length:    { bsonType: ["int", "null"] },
                            format:        { bsonType: ["string", "null"] },
                            is_pii:        { bsonType: "bool" },
                            pii_classification: { bsonType: ["string", "null"] }
                          } } },
        created_by:     { bsonType: "string" },
        created_at:     { bsonType: "date" }
      }
    }},
    validationLevel: "strict", validationAction: "error"
  });
}
db.schema_versions.createIndex({ schema_id: 1 }, { unique: true });
db.schema_versions.createIndex({ file_type_id: 1, version: 1 }, { unique: true });
// DB-enforced invariant from the LLD: exactly ONE active schema per file type
db.schema_versions.createIndex(
  { file_type_id: 1 },
  { unique: true, partialFilterExpression: { status: "active" } }
);

// ---------- validation_rules ----------
if (!db.getCollectionInfos({name: "validation_rules"}).length) {
  db.createCollection("validation_rules", {
    validator: { $jsonSchema: {
      bsonType: "object",
      required: ["rule_id", "rule_code", "category", "severity", "params", "priority", "is_active", "version", "created_by", "created_at"],
      properties: {
        rule_id:    { bsonType: "string", pattern: "^rule_" },
        rule_code:  { bsonType: "string", minLength: 3 },
        category:   { enum: ["FILE", "HEADER", "CONTENT"] },
        severity:   { enum: ["ERROR", "WARN"] },
        applies_to: { bsonType: ["string", "null"] },   // ft_* or null = global
        params:     { bsonType: "object" },
        priority:   { bsonType: "int", minimum: 1 },
        is_active:  { bsonType: "bool" },
        version:    { bsonType: "int", minimum: 1 },
        created_by: { bsonType: "string" },
        created_at: { bsonType: "date" },
        updated_at: { bsonType: ["date", "null"] }
      }
    }},
    validationLevel: "strict", validationAction: "error"
  });
}
db.validation_rules.createIndex({ rule_id: 1 }, { unique: true });
db.validation_rules.createIndex({ rule_code: 1, version: 1 }, { unique: true });
```

## 3. DDL — `routing_policies`, `volume_map`

```javascript
// ---------- routing_policies ----------
if (!db.getCollectionInfos({name: "routing_policies"}).length) {
  db.createCollection("routing_policies", {
    validator: { $jsonSchema: {
      bsonType: "object",
      required: ["policy_id", "name", "priority", "match", "action", "is_active"],
      properties: {
        policy_id: { bsonType: "string", pattern: "^pol_" },
        name:     { bsonType: "string", minLength: 1 },
        priority: { bsonType: "int", minimum: 1 },
        match:    { bsonType: "object" },
        action:   { enum: ["MOVE_INTAKE", "MOVE_QUARANTINE", "REJECT", "NOTIFY_ONLY"] },
        is_active:{ bsonType: "bool" }
      }
    }},
    validationLevel: "strict", validationAction: "error"
  });
}
db.routing_policies.createIndex({ policy_id: 1 }, { unique: true });
db.routing_policies.createIndex({ priority: 1 });

// ---------- volume_map ----------
if (!db.getCollectionInfos({name: "volume_map"}).length) {
  db.createCollection("volume_map", {
    validator: { $jsonSchema: {
      bsonType: "object",
      required: ["map_id", "file_type_id", "destination_type", "volume_root", "subfolder", "naming_convention"],
      properties: {
        map_id:            { bsonType: "string", pattern: "^vol_" },
        file_type_id:      { bsonType: "string" },   // ft_* or "*"
        destination_type:  { enum: ["STAGING", "INTAKE", "QUARANTINE", "REJECT"] },
        volume_root:       { bsonType: "string", minLength: 1 },
        subfolder:         { bsonType: "string" },
        naming_convention:  { bsonType: "string", minLength: 1 }
      }
    }},
    validationLevel: "strict", validationAction: "error"
  });
}
db.volume_map.createIndex({ map_id: 1 }, { unique: true });
db.volume_map.createIndex({ file_type_id: 1, destination_type: 1 }, { unique: true });
```

## 4. Seeds — registry contracts (R1 review targets)

```javascript
use fvr
const NOW = new Date();

// ---------- source_systems ----------
db.source_systems.insertMany([
  { source_id: "src_portal", name: "UI Portal (React)", source_type: "PORTAL",
    connection_ref: "internal:portal", poll_or_push: "PUSH",
    detection_config: {}, is_active: true, created_at: NOW }
], { ordered: false });

// ---------- file_types ---------- // allowed_extensions / size limits are R1 targets
db.file_types.insertMany([
  { file_type_id: "ft_customer", code: "customer", description: "Customer master",
    allowed_extensions: [".csv"], max_file_size_mb: 500, min_rows: 1,
    encoding: "utf-8", delimiter: ",", header_row: true,
    identification: { patterns: ["^cust(omer)?[_-].*\\.csv$"], header_fingerprint: [] },
    is_active: true, version: 1 },
  { file_type_id: "ft_order",    code: "order",    description: "Orders",
    allowed_extensions: [".csv"], max_file_size_mb: 500, min_rows: 1,
    encoding: "utf-8", delimiter: ",", header_row: true,
    identification: { patterns: ["^orders?[_-].*\\.csv$"], header_fingerprint: [] },
    is_active: true, version: 1 },
  { file_type_id: "ft_payments",  code: "payments",  description: "Payments (MDD billing domain)",
    allowed_extensions: [".csv"], max_file_size_mb: 500, min_rows: 1,
    encoding: "utf-8", delimiter: ",", header_row: true,
    identification: { patterns: ["^payments?[_-].*\\.csv$"], header_fingerprint: [] },
    is_active: true, version: 1 },
  { file_type_id: "ft_product",  code: "product",  description: "Product catalog",
    allowed_extensions: [".csv"], max_file_size_mb: 500, min_rows: 1,
    encoding: "utf-8", delimiter: ",", header_row: true,
    identification: { patterns: ["^product[s]?[_-].*\\.csv$"], header_fingerprint: [] },
    is_active: true, version: 1 },
  { file_type_id: "ft_employee", code: "employee", description: "Employee roster",
    allowed_extensions: [".csv"], max_file_size_mb: 500, min_rows: 1,
    encoding: "utf-8", delimiter: ",", header_row: true,
    identification: { patterns: ["^employee[s]?[_-].*\\.csv$"], header_fingerprint: [] },
    is_active: true, version: 1 }
], { ordered: false });

// ---------- schema_versions ---------- // REAL contracts from mdd_platform silver: src_001, src_002, src_003, src_005, src_004
// Business columns only (no _ingest_ts/_merged_ts); logical data_types for validation; PII aligned with existing mdd_platform column masks
const F = (defs) => defs.map((d, i) => ({ order: i + 1, name: d[0], data_type: d[1],
  nullable: d[2], max_length: null, format: null, is_pii: !!d[3],
  pii_classification: d[3] || null }));
const SCH = (id, ft, defs) => ({ schema_id: id, file_type_id: ft, version: 1, status: "active",
  effective_from: NOW, fields: F(defs), created_by: "system:seed", created_at: NOW });
db.schema_versions.insertMany([
  SCH("sch_customer_v1", "ft_customer", [            // src_001_customers / src_006_customers — 10 cols
    ["customer_id", "string", false], ["first_name", "string", false, "PII"],
    ["last_name", "string", false, "PII"], ["email", "string", true, "PII"],
    ["phone", "string", true, "PII"], ["address", "string", true, "PII"],
    ["city", "string", true], ["country", "string", true],
    ["signup_date", "date", false], ["loyalty_tier", "string", true] ]),
  SCH("sch_order_v1", "ft_order", [               // src_002_orders / src_007_orders — 7 cols
    ["order_id", "string", false], ["customer_id", "string", false], ["order_date", "date", false],
    ["amount", "decimal", false], ["currency", "string", false],
    ["status", "string", true], ["channel", "string", true] ]),
  SCH("sch_payments_v1", "ft_payments", [          // src_003_payments / src_008_payments — 7 cols
    ["payment_id", "string", false], ["order_id", "string", false], ["payment_method", "string", false],
    ["amount", "decimal", false], ["card_last4", "string", true, "PII"],
    ["payment_date", "date", false], ["status", "string", true] ]),
  SCH("sch_product_v1", "ft_product", [            // src_005_products / src_010_products — 7 cols
    ["product_id", "string", false], ["product_name", "string", false], ["category", "string", true],
    ["price", "decimal", false], ["stock_qty", "integer", false],
    ["supplier_id", "string", true], ["launch_date", "date", true] ]),
  SCH("sch_employee_v1", "ft_employee", [          // src_004_employees / src_009_employees — 7 cols
    ["employee_id", "string", false], ["full_name", "string", false, "PII"],
    ["email", "string", true, "PII"], ["phone", "string", true, "PII"],
    ["department", "string", true], ["hire_date", "date", true], ["salary", "decimal", true, "PII"] ])
], { ordered: false });

// ---------- validation_rules ---------- // the 8 Phase-1 seeds (LLD §10.2)
const RULE = (id, code, cat, sev, prio, params, applies) => ({
  rule_id: id, rule_code: code, category: cat, severity: sev,
  applies_to: applies || null, params, priority: prio,
  is_active: true, version: 1, created_by: "system:seed", created_at: NOW });
db.validation_rules.insertMany([
  RULE("rule_file_type_allowed",    "FILE_TYPE_ALLOWED",    "FILE",   "ERROR", 10, { enforce_declared_extension: true }),
  RULE("rule_file_size_max",        "FILE_SIZE_MAX",        "FILE",   "ERROR", 20, { max_mb: 500 }),
  RULE("rule_file_not_empty",       "FILE_NOT_EMPTY",       "FILE",   "ERROR", 30, { min_bytes: 1 }),
  RULE("rule_file_not_duplicate",   "FILE_NOT_DUPLICATE",   "FILE",   "ERROR", 40, { scope: "global" }),
  RULE("rule_header_required",      "HEADER_REQUIRED",      "HEADER", "ERROR", 10, {}),
  RULE("rule_schema_match",         "SCHEMA_MATCH",         "HEADER", "ERROR", 20, { mode: "exact_names" }),
  RULE("rule_column_order_match",   "COLUMN_ORDER_MATCH",   "HEADER", "ERROR", 30, { mode: "strict" }),
  RULE("rule_file_not_empty_rows",  "FILE_NOT_EMPTY_ROWS",  "CONTENT","WARN",  10, { min_rows: 1 })
], { ordered: false });

// ---------- routing_policies ---------- // first-match by priority (ascending)
db.routing_policies.insertMany([
  { policy_id: "pol_error_quarantine",   name: "Any ERROR verdict → Quarantine",  priority: 10,
    match: { any_error: true },                       action: "MOVE_QUARANTINE", is_active: true },
  { policy_id: "pol_unidentified_quarantine", name: "Unidentified → Quarantine", priority: 20,
    match: { status: "UNIDENTIFIED" },               action: "MOVE_QUARANTINE", is_active: true },
  { policy_id: "pol_passed_intake",      name: "PASSED/WARNED → Intake",        priority: 30,
    match: { status: { $in: ["PASSED", "WARNED"] } }, action: "MOVE_INTAKE",     is_active: true }
], { ordered: false });

// ---------- volume_map ---------- // EDIT volume_root paths (open question #5)
const VM = (id, ft, dest, root, sub) => ({ map_id: id, file_type_id: ft, destination_type: dest,
  volume_root: root, subfolder: sub, naming_convention: "{file_type}_{file_id}_{ts}.ext" });
const INTAKE = "/Volumes/<catalog>/<schema>/intake";    // EDIT
const QUAR  = "/Volumes/<catalog>/<schema>/quarantine";  // EDIT
const STAGING = "/Volumes/<catalog>/<schema>/staging";  // EDIT (Storage Addendum — uploads stream here)
db.volume_map.insertMany([
  VM("vol_customer_intake", "ft_customer", "INTAKE", INTAKE, "customer"),
  VM("vol_order_intake",    "ft_order",    "INTAKE", INTAKE, "order"),
  VM("vol_payments_intake",  "ft_payments",  "INTAKE", INTAKE, "payments"),
  VM("vol_product_intake",  "ft_product",  "INTAKE", INTAKE, "product"),
  VM("vol_employee_intake", "ft_employee", "INTAKE", INTAKE, "employee"),
  VM("vol_catchall_quarantine", "*",       "QUARANTINE", QUAR, "unidentified"),
  // staging catch-all (Storage Addendum): uploads stream here per session; original names kept
  { map_id: "vol_staging_catchall", file_type_id: "*", destination_type: "STAGING",
    volume_root: STAGING, subfolder: "sessions", naming_convention: "{original_name}" }
], { ordered: false });

// ---------- users ---------- // one-time bootstrap admin — EDIT email, then comment out
// db.users.insertOne({ user_id: "usr_bootstrap_admin", email: "admin@yourcompany.com",
//   display_name: "Bootstrap Admin", roles: ["ADMIN", "OPS", "UPLOADER"],
//   department: null, is_active: true, created_at: NOW, last_login_at: null });
```

> **R1 review targets inside the seeds:** the 5 file types (+extensions/limits), the 5 schema column lists (**now the real contracts, sourced from mdd_platform silver schemas — src_001 customers, src_002 orders, src_003 payments, src_005 products, src_004 employees**; logical types assigned for validation, PII flags aligned with the existing mdd_platform column masks, `billing` renamed `payments` to match the real domain), the 8 rules (codes, severities, params, priorities), the 3 routing policies, and the volume roots. Anything you change here is a document edit — never a code change.

## 5. Verification — run after cells 1–4

```javascript
use fvr

// Expected counts: 1 / 5 / 5 / 8 / 3 / 7 (users: 0 until first login or bootstrap)
["source_systems","file_types","schema_versions","validation_rules","routing_policies","volume_map","users"]
  .forEach(c => print(c.padEnd(20), db[c].countDocuments({})));

// Invariant: exactly one ACTIVE schema per file type (partial unique index enforces it)
db.schema_versions.aggregate([
  { $match: { status: "active" } },
  { $group: { _id: "$file_type_id", active_versions: { $sum: 1 } } },
  { $match: { active_versions: { $ne: 1 } } }              // expect: no rows
]).forEach(d => print("VIOLATION:", JSON.stringify(d)));

// Rules by execution phase — the engine's run order
db.validation_rules.find({}, { _id: 0, rule_id: 1, category: 1, severity: 1, priority: 1 })
  .sort({ category: 1, priority: 1 })
  .forEach(d => print(JSON.stringify(d)));

// Routing decision table — first match by priority wins
printjson(db.routing_policies.find({}, { _id: 0 }).sort({ priority: 1 }).toArray());

// Volume destinations as the transfer worker will resolve them
printjson(db.volume_map.find({}, { _id: 0 }).sort({ destination_type: 1, subfolder: 1 }).toArray());

// All validators active on the registry collections
["users","source_systems","file_types","schema_versions","validation_rules","routing_policies","volume_map"]
  .forEach(c => print(c.padEnd(20), db.getCollectionInfos({ name: c })[0].options.validationAction));
```

**Pass criteria (R1):** counts match the table above; no active-schema violations; rules print in FILE → HEADER → CONTENT order; every volume row has a real path you confirmed; all 7 collections report `validationAction: error`.